## 2 卷积和池化层

### 2.1 理论计算题

### （1）计算卷积层输出特征图尺寸

输出高宽计算公式：

$$
O=\left\lfloor \frac{N+2P-K}{S}\right\rfloor+1
$$

代入数据：

$$
O=\left\lfloor \frac{32+2\times2-5}{2}\right\rfloor+1
=\left\lfloor \frac{31}{2}\right\rfloor+1
=15+1
=16
$$

卷积核个数为 16，因此输出通道数为 16。

故输出特征图尺寸为：

$$
\boxed{16\times16\times16}
$$

（通道数 × 高 × 宽）


### （2）计算单个输出像素所需乘法次数

卷积核尺寸为：

$$
3\times5\times5
$$

因此一个输出像素需要的乘法次数为：

$$
3\times5\times5=75
$$

故答案为：

$$
\boxed{75}
$$

### 2.2 编程题

In [7]:
import numpy as np

def max_pool2d(x, kernel_size=2, stride=2, padding=0):
    """
    二维最大池化

    参数：
        x : 输入特征图 (H, W)
        kernel_size : 池化窗口大小
        stride : 步幅
        padding : 填充大小

    返回：
        output : 池化后的特征图
    """

    # Padding
    if padding > 0:
        x = np.pad(
            x,
            ((padding, padding), (padding, padding)),
            mode='constant',
            constant_values=0
        )

    H, W = x.shape

    out_h = (H - kernel_size) // stride + 1
    out_w = (W - kernel_size) // stride + 1

    output = np.zeros((out_h, out_w))

    for i in range(out_h):
        for j in range(out_w):
            h_start = i * stride
            h_end = h_start + kernel_size

            w_start = j * stride
            w_end = w_start + kernel_size

            window = x[h_start:h_end, w_start:w_end]

            output[i, j] = np.max(window)

    return output


# 测试
x = np.array([
    [1, 3, 2, 4],
    [5, 6, 1, 2],
    [7, 2, 8, 3],
    [1, 4, 2, 9]
])

result = max_pool2d(
    x,
    kernel_size=2,
    stride=2,
    padding=0
)

print("输入：")
print(x)

print("\n池化结果：")
print(result)

输入：
[[1 3 2 4]
 [5 6 1 2]
 [7 2 8 3]
 [1 4 2 9]]

池化结果：
[[6. 4.]
 [7. 9.]]


## 3 LeNet, AlexNet, VGG 和 NiN
### 3.1 理论计算题

### （1）计算一个 5×5 卷积层的参数量

输入通道数和输出通道数均为 $C$，卷积核大小为 $5\times5$。

参数量为：

$$
C \times C \times 5 \times 5
$$

即：

$$
\boxed{25C^2}
$$



### （2）计算两个串联的 3×3 卷积层的总参数量

单个 $3\times3$ 卷积层参数量为：

$$
C \times C \times 3 \times 3
= 9C^2
$$

两个卷积层总参数量为：

$$
2 \times 9C^2
= 18C^2
$$

即：

$$
\boxed{18C^2}
$$



### 3.2 编程题

In [8]:
import torch
import torch.nn as nn

def NiNBlock(in_channels, out_channels,
             kernel_size, stride, padding):

    return nn.Sequential(
        nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding
        ),
        nn.ReLU(),

        nn.Conv2d(
            in_channels=out_channels,
            out_channels=out_channels,
            kernel_size=1
        ),
        nn.ReLU(),

        nn.Conv2d(
            in_channels=out_channels,
            out_channels=out_channels,
            kernel_size=1
        ),
        nn.ReLU()
    )


# 测试
x = torch.randn(1, 3, 224, 224)

nin_block = NiNBlock(
    in_channels=3,
    out_channels=96,
    kernel_size=11,
    stride=4,
    padding=0
)

y = nin_block(x)

print("输出尺寸：", y.shape)

输出尺寸： torch.Size([1, 96, 54, 54])


## 4 Inception, 批量归一化和残差网络
### 4.1 理论题


**已知：**  
输入样本：$$x_1 = 2, x_2 = 4, x_3 = 6, x_4 = 8$$  
参数：$$\gamma = 2, \beta = 1, \epsilon = 0$$


**1. 计算均值：**

$$
\mu = \frac{2 + 4 + 6 + 8}{4} = 5
$$

**2. 计算方差：**

$$
\sigma^2 = \frac{(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2}{4} = 5
$$

**3. 计算归一化值：**

$$
\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} = \frac{x_i - 5}{\sqrt{5}}
$$

**4. 最终结果：**  
$$y_i = \gamma \hat{x}_i + \beta = 2\hat{x}_i + 1$$

$$
y_1 = 2 \times \frac{-3}{\sqrt{5}} + 1 = 1 - \frac{6}{\sqrt{5}}
$$

$$
y_2 = 2 \times \frac{-1}{\sqrt{5}} + 1 = 1 - \frac{2}{\sqrt{5}}
$$

$$
y_3 = 2 \times \frac{1}{\sqrt{5}} + 1 = 1 + \frac{2}{\sqrt{5}}
$$

$$
y_4 = 2 \times \frac{3}{\sqrt{5}} + 1 = 1 + \frac{6}{\sqrt{5}}
$$


**所以：**

$$
y_1 = 1 - \frac{6}{\sqrt{5}}, \quad y_2 = 1 - \frac{2}{\sqrt{5}}, \quad y_3 = 1 + \frac{2}{\sqrt{5}}, \quad y_4 = 1 + \frac{6}{\sqrt{5}}
$$

### 4.2 编程题

In [9]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        
        # ---------- 主路径：两个 3x3 卷积 + BN ----------
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # ---------- 侧路径（残差连接） ----------
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.conv3 = None
    
    def forward(self, X):
        # 主路径
        Y = self.conv1(X)
        Y = self.bn1(Y)
        Y = torch.relu(Y)  # 第一个 BN 后的激活函数
        Y = self.conv2(Y)
        Y = self.bn2(Y)
        
        # 侧路径（残差连接）
        if self.conv3:
            X = self.conv3(X)
        
        # 按元素相加（f(x) + x）
        Y += X
        return torch.relu(Y)  # 相加后的最终激活

In [10]:
# 测试：in_channels=3, out_channels=3 (不使用1x1)
block1 = Residual(3, 3, use_1x1conv=False)
x = torch.randn(1, 3, 32, 32)
print("Output shape (no 1x1):", block1(x).shape)  # 应输出 torch.Size([1, 3, 32, 32])

# 测试：in_channels=3, out_channels=64 (使用1x1)
block2 = Residual(3, 64, use_1x1conv=True, stride=2)
x = torch.randn(1, 3, 32, 32)
print("Output shape (with 1x1):", block2(x).shape)  # 应输出 torch.Size([1, 64, 16, 16])

Output shape (no 1x1): torch.Size([1, 3, 32, 32])
Output shape (with 1x1): torch.Size([1, 64, 16, 16])


## 5 图像增广，微调和样式迁移
### 5.1 理论题



#### 1. 为什么我们通常对除了最终输出层之外的“底层特征提取层”设置较小的学习率（甚至将其参数固定/冻结），而对新初始化的“顶层输出层”设置较大的学习率？

这是因为在大型源数据集（如 ImageNet）上预训练得到的模型，其底层特征提取层已经学习到了通用的视觉特征（如边缘、纹理、形状等），这些特征对于大多数视觉任务都是有效的。因此，在微调时我们不需要大幅改变这些通用特征，只需对其进行微调即可。如果对底层设置较大的学习率，可能会破坏这些已经学好的特征，导致过拟合或训练不稳定。而对于新初始化的顶层输出层，由于它是随机初始化的，对目标任务的类别信息毫无先验知识，需要快速适应新任务，因此需要设置较大的学习率来加速其收敛。

#### 2. 如果目标数据集非常小，且与源数据集非常相似，我们应该采取什么样的微调策略以防止过拟合？

当目标数据集非常小且与源数据集非常相似时，最有效的微调策略是冻结底层特征提取层（即将其参数固定，不参与梯度更新），仅训练顶层的输出层。这是因为源数据集已经学习到了与目标数据集高度相关的通用特征，完全保留这些特征可以避免在小数据集上因参数过多而导致的过拟合问题。同时，仅训练顶层输出层可以显著减少可训练参数量，从而在保持模型性能的同时有效防止过拟合。

### 5.2 编程题

In [11]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.RandomResizedCrop(
        224, scale=(0.08, 1.0)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.5,
        contrast=0.5,
        saturation=0.5
    ),
    transforms.ToTensor()
])

## 6 目标检测，计算机视觉训练技巧
### 6.1 理论题

### 6.1 理论计算题解答

**已知：**
真实框 $$A = [10, 10, 50, 50]$$
预测框 $$B = [30, 30, 70, 70]$$
格式：[左上角 x, 左上角 y, 右下角 x, 右下角 y]

#### 1. 计算交集面积

交集矩形左上角：
$$
x_{\min} = \max(10, 30) = 30
$$
$$
y_{\min} = \max(10, 30) = 30
$$

交集矩形右下角：
$$
x_{\max} = \min(50, 70) = 50
$$
$$
y_{\max} = \min(50, 70) = 50
$$

交集宽和高：
$$
\text{宽} = 50 - 30 = 20
$$
$$
\text{高} = 50 - 30 = 20
$$

交集面积：
$$
\text{Intersection} = 20 \times 20 = 400
$$


#### 2. 计算并集面积

框 A 面积：
$$
\text{Area}_A = (50-10) \times (50-10) = 40 \times 40 = 1600
$$

框 B 面积：
$$
\text{Area}_B = (70-30) \times (70-30) = 40 \times 40 = 1600
$$

并集面积：
$$
\text{Union} = 1600 + 1600 - 400 = 2800
$$


#### 3. 计算 IoU

$$
\text{IoU} = \frac{\text{Intersection}}{\text{Union}} = \frac{400}{2800} = \frac{1}{7}
$$


#### 答案：

$$
\text{IoU} = \frac{1}{7} \approx 0.1429
$$

### 6.2 编程题

In [12]:
import torch
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, targets, eps=0.1):
    """
    logits: (batch_size, num_classes)
    targets: (batch_size,) 真实类别索引
    eps: label smoothing factor
    """

    num_classes = logits.size(1)

    # log-probabilities
    log_probs = F.log_softmax(logits, dim=1)

    # 构造平滑后的标签分布
    with torch.no_grad():
        true_dist = torch.zeros_like(log_probs)
        true_dist.fill_(eps / (num_classes - 1))
        true_dist.scatter_(1, targets.unsqueeze(1), 1 - eps)

    # 计算交叉熵
    loss = -torch.sum(true_dist * log_probs, dim=1).mean()

    return loss